In [4]:
import pandas as pd


df = pd.read_csv('final_df.csv')

In [ ]:
import pandas as pd
import numpy as np
import re
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

# ---- config ----
target_col = 'import_insulin, Insulin [Units/volume] in Serum o'
exclude_cols = ['participant_id', 'study_group', 'split', target_col, 'race', 'sex']

# XGBoost rejects feature names containing [, ], or < (e.g. units like "[Mass/volume]")
def sanitize(col):
    return re.sub(r'[\[\]<]', '', col)

df = df.rename(columns={c: sanitize(c) for c in df.columns})
target_col = sanitize(target_col)
exclude_cols = [sanitize(c) for c in exclude_cols]

feature_cols = [c for c in df.columns if c not in exclude_cols]

study_groups = df['study_group'].unique()

param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

results = {}  # study_group -> dict of model, shap_values, X_test, etc.

for group in study_groups:
    print(f"\n{'='*50}\n{group}\n{'='*50}")

    group_df = df[df['study_group'] == group]
    train_mask = group_df['split'] == 'train'
    test_mask = group_df['split'] == 'test'

    X_train = group_df.loc[train_mask, feature_cols]
    y_train = group_df.loc[train_mask, target_col]
    X_test = group_df.loc[test_mask, feature_cols]
    y_test = group_df.loc[test_mask, target_col]

    print(f"train: {X_train.shape}, test: {X_test.shape}")

    if len(X_train) < 20 or len(X_test) < 5:
        print(f"skipping {group}, too small")
        continue

    base_model = xgb.XGBRegressor(random_state=42, n_jobs=1)  # n_jobs=1 here since GridSearchCV parallelizes outer loop

    grid = GridSearchCV(
        base_model,
        param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    train_r2 = r2_score(y_train, best_model.predict(X_train))
    test_r2 = r2_score(y_test, best_model.predict(X_test))

    print(f"best params: {grid.best_params_}")
    print(f"train R2: {train_r2:.3f}, test R2: {test_r2:.3f}")

    # ---- SHAP for this group's best model ----
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer(X_test)

    results[group] = {
        'model': best_model,
        'grid': grid,
        'shap_values': shap_values,
        'X_test': X_test,
        'y_test': y_test,
        'train_r2': train_r2,
        'test_r2': test_r2
    }

# ---- plot SHAP summary per group ----
for group, res in results.items():
    print(f"\n{group} (test R2={res['test_r2']:.3f})")
    shap.summary_plot(res['shap_values'], res['X_test'], show=True)